# Simulation And Analysis

Monte Carlo-style path simulation notebook for cross-exchange arbitrage execution.

This notebook consumes cached artifacts produced by `data_pipeline.ipynb` and evaluates
strategy behavior under fees, slippage, latency, inventory, and balance constraints.

## Required Inputs
- Composite pages cache: `persist/analysis-cache/composite_pages_btcusdt.pkl.gz`.
- Candidate cache: `persist/analysis-cache/cross_exchange_arbitrage_btcusdt2.pkl.gz`.

## Outputs
- `sim_paths_df`: path-level event log (executions/skips/PnL over time).
- `sim_balances_df`: final balances per path/venue (with initial balance snapshot).
- Yield curves and path summary tables.

## Recommended Workflow
1. Rebuild data caches in `data_pipeline.ipynb`.
2. Set simulation config.
3. Run full notebook top-to-bottom.
4. Compare path summary and yield curves across scenario changes.



# Arbitrage Path Simulator

Config-driven simulator for cross-exchange arbitrage using cached unified orderbook pages and cached candidate events.

## Simulation Config Guide

Key settings in `SIM_CONFIG`:
- **Path sampling**: `n_paths`, `base_seed`, `latency_model`.
- **Cost model**: `fee_bps_by_venue`, `slippage_bps_per_side`, `fixed_cost_quote`, `latency_haircut_bps`.
- **Trade gate**: `min_net_edge_quote`, `max_candidate_age_ms`.
- **Inventory constraints**: `allow_negative_balances`, `initial_balances`.
- **Initial balance mode**:
  - `initial_balance_mode="fixed"`: identical starting balances for every path.
  - `initial_balance_mode="vary_by_path"`: randomized start by path using `initial_balance_jitter_pct`.
- **Strategy block**:
  - `all_at_once`: single fill attempt up to `max_trade_qty`.
  - `partitioned`: sequential slices by `partition_count` / `partition_step_ms`.

Interpretation tips:
- Flat yield curves usually mean all candidates are skipped or net-negative after costs.
- Check status counts (`executed`, `skipped_*`) before tuning strategy logic.



## Config

Set portfolio, execution, latency, and strategy parameters here.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import math
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PAGES_CACHE_PATH = Path("/home/fkonrad97/projects/pop/persist/analysis-cache/composite_pages_btcusdt.pkl.gz")
CANDIDATES_CACHE_PATH = Path("/home/fkonrad97/projects/pop/persist/analysis-cache/cross_exchange_arbitrage_btcusdt2.pkl.gz")

SIM_CONFIG = {
    "symbol": "BTCUSDT",
    "allow_negative_balances": False,
    "n_paths": 25,
    "base_seed": 42,
    "fee_bps_by_venue": {
        "binance": 10.0,
        "okx": 10.0,
        "bybit": 10.0,
        "kucoin": 10.0,
    },
    "slippage_bps_per_side": 0.0,
    "fixed_cost_quote": 0.0,
    "latency_haircut_bps": 0.0,
    "min_net_edge_quote": 0.0,
    "max_candidate_age_ms": None,
    "initial_balances": {
        "binance": {"BTC": 0.05, "USDT": 5000.0},
        "okx": {"BTC": 0.05, "USDT": 5000.0},
        "bybit": {"BTC": 0.05, "USDT": 5000.0},
        "kucoin": {"BTC": 0.05, "USDT": 5000.0},
    },
    "initial_balance_mode": "fixed",  # fixed | vary_by_path
    "initial_balance_jitter_pct": 0.0,  # only used when mode=vary_by_path (e.g. 0.20 => +/-20%)
    "latency_model": {
        "kind": "uniform",  # uniform | normal | lognormal
        "uniform_min_ms": 10.0,
        "uniform_max_ms": 80.0,
        "normal_mean_ms": 40.0,
        "normal_std_ms": 15.0,
        "lognormal_mean_ln": 3.2,
        "lognormal_sigma_ln": 0.35,
        "clip_min_ms": 1.0,
        "clip_max_ms": 300.0,
    },
    "strategy": {
        "name": "all_at_once",  # all_at_once | partitioned
        "max_trade_qty": 0.01,
        "partition_count": 4,
        "partition_step_ms": 20.0,
    },
}


## Data Loading

In [ ]:
def load_table_auto(path: Path) -> pd.DataFrame:
    """Load cache from .pkl/.pkl.gz or .csv."""
    if not path.exists():
        raise FileNotFoundError(path)
    lower = path.name.lower()
    if lower.endswith('.csv'):
        return pd.read_csv(path)
    if lower.endswith('.pkl') or lower.endswith('.pkl.gz'):
        compression = 'gzip' if lower.endswith('.gz') else None
        return pd.read_pickle(path, compression=compression)
    raise ValueError(f"Unsupported cache file extension: {path}")


def normalize_candidate_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize candidate column names and validate required simulation fields."""
    if df.empty:
        return df.copy()
    out = df.copy()
    alias_map = {
        'buy_exchange': 'buy_venue',
        'sell_exchange': 'sell_venue',
        'buyVenue': 'buy_venue',
        'sellVenue': 'sell_venue',
        'buyPx': 'buy_price',
        'sellPx': 'sell_price',
        'buyQty': 'buy_qty',
        'sellQty': 'sell_qty',
    }
    for src, dst in alias_map.items():
        if src in out.columns and dst not in out.columns:
            out[dst] = out[src]

    required = [
        'ts_book_ns',
        'buy_venue',
        'sell_venue',
        'buy_price',
        'sell_price',
        'buy_qty',
        'sell_qty',
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(
            'Candidate cache schema mismatch. Missing columns: '
            f"{missing}. Available columns: {list(out.columns)}"
        )

    numeric_cols = ['ts_book_ns', 'buy_price', 'sell_price', 'buy_qty', 'sell_qty']
    for c in numeric_cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.dropna(subset=numeric_cols)

    if 'buy_age_ms' not in out.columns:
        out['buy_age_ms'] = 0.0
    if 'sell_age_ms' not in out.columns:
        out['sell_age_ms'] = 0.0

    out['buy_age_ms'] = pd.to_numeric(out['buy_age_ms'], errors='coerce').fillna(0.0)
    out['sell_age_ms'] = pd.to_numeric(out['sell_age_ms'], errors='coerce').fillna(0.0)
    out['buy_venue'] = out['buy_venue'].astype(str)
    out['sell_venue'] = out['sell_venue'].astype(str)
    out['ts_book_ns'] = out['ts_book_ns'].astype('int64')
    return out


pages_df = load_table_auto(PAGES_CACHE_PATH)
candidates_df_raw = load_table_auto(CANDIDATES_CACHE_PATH)
candidates_df = normalize_candidate_schema(candidates_df_raw)

if 'ts_book_dt' in pages_df.columns:
    pages_df['ts_book_dt'] = pd.to_datetime(pages_df['ts_book_dt'])
if 'ts_book_dt' in candidates_df.columns:
    candidates_df['ts_book_dt'] = pd.to_datetime(candidates_df['ts_book_dt'])

pages_df = pages_df.sort_values('ts_book_ns').reset_index(drop=True)
candidates_df = candidates_df.sort_values('ts_book_ns').reset_index(drop=True)

{
    'pages_rows': int(len(pages_df)),
    'candidates_rows': int(len(candidates_df)),
    'candidate_columns': list(candidates_df.columns),
    'pages_range': [int(pages_df.iloc[0]['ts_book_ns']) if len(pages_df) else None, int(pages_df.iloc[-1]['ts_book_ns']) if len(pages_df) else None],
    'cand_range': [int(candidates_df.iloc[0]['ts_book_ns']) if len(candidates_df) else None, int(candidates_df.iloc[-1]['ts_book_ns']) if len(candidates_df) else None],
}


## Engine Types

In [ ]:
@dataclass
class VenueBalance:
    base: float
    quote: float


@dataclass
class SimTrade:
    path_id: int
    ts_book_ns: int
    ts_exec_ns: int
    buy_venue: str
    sell_venue: str
    qty: float
    buy_px_eff: float
    sell_px_eff: float
    net_edge_quote: float
    status: str


def fee_rate(venue: str, cfg: dict) -> float:
    return float(cfg["fee_bps_by_venue"].get(venue, 0.0)) / 10000.0


def sample_latency_ms(rng: random.Random, cfg: dict) -> float:
    m = cfg["latency_model"]
    kind = m["kind"]
    if kind == "uniform":
        x = rng.uniform(float(m["uniform_min_ms"]), float(m["uniform_max_ms"]))
    elif kind == "normal":
        x = rng.gauss(float(m["normal_mean_ms"]), float(m["normal_std_ms"]))
    elif kind == "lognormal":
        x = rng.lognormvariate(float(m["lognormal_mean_ln"]), float(m["lognormal_sigma_ln"]))
    else:
        raise ValueError(f"Unsupported latency model: {kind}")
    return max(float(m["clip_min_ms"]), min(float(m["clip_max_ms"]), x))


def effective_prices(row: pd.Series, cfg: dict) -> tuple[float, float]:
    slippage = float(cfg["slippage_bps_per_side"]) / 10000.0
    buy_px = float(row["buy_price"]) * (1.0 + slippage)
    sell_px = float(row["sell_price"]) * (1.0 - slippage)
    buy_px *= (1.0 + fee_rate(str(row["buy_venue"]), cfg))
    sell_px *= (1.0 - fee_rate(str(row["sell_venue"]), cfg))
    return buy_px, sell_px


def find_page_at_or_after(ts_ns: int, pages: pd.DataFrame) -> pd.Series | None:
    idx = pages["ts_book_ns"].searchsorted(ts_ns, side="left")
    if idx >= len(pages):
        return None
    return pages.iloc[int(idx)]


## Strategy Logic

In [ ]:
def feasible_qty(row: pd.Series, balances: dict[str, VenueBalance], cfg: dict, max_qty: float) -> float:
    buy_v = str(row["buy_venue"])
    sell_v = str(row["sell_venue"])
    buy_px_eff, _ = effective_prices(row, cfg)
    top_qty = min(float(row["buy_qty"]), float(row["sell_qty"]), max_qty)
    if top_qty <= 0:
        return 0.0

    if cfg["allow_negative_balances"]:
        return top_qty

    buy_cap = balances[buy_v].quote / max(buy_px_eff, 1e-12)
    sell_cap = balances[sell_v].base
    return max(0.0, min(top_qty, buy_cap, sell_cap))


def apply_trade(row: pd.Series, qty: float, balances: dict[str, VenueBalance], cfg: dict) -> tuple[float, float, float]:
    buy_v = str(row["buy_venue"])
    sell_v = str(row["sell_venue"])
    buy_px_eff, sell_px_eff = effective_prices(row, cfg)
    gross_buy = buy_px_eff * qty
    gross_sell = sell_px_eff * qty
    mid = (float(row["buy_price"]) + float(row["sell_price"])) / 2.0
    haircut = mid * qty * (float(cfg["latency_haircut_bps"]) / 10000.0)
    net = gross_sell - gross_buy - haircut - float(cfg["fixed_cost_quote"])

    balances[buy_v].quote -= gross_buy
    balances[buy_v].base += qty
    balances[sell_v].base -= qty
    balances[sell_v].quote += gross_sell
    return buy_px_eff, sell_px_eff, net


def execute_all_at_once(row: pd.Series, exec_page: pd.Series, balances: dict[str, VenueBalance], cfg: dict) -> tuple[float, float, float, str]:
    max_qty = float(cfg["strategy"]["max_trade_qty"])
    qty = feasible_qty(row, balances, cfg, max_qty)
    if qty <= 0:
        return 0.0, 0.0, 0.0, "skipped_no_capacity"
    buy_px_eff, sell_px_eff, net = apply_trade(row, qty, balances, cfg)
    if net < float(cfg["min_net_edge_quote"]):
        return qty, buy_px_eff, sell_px_eff, "skipped_below_threshold"
    return qty, buy_px_eff, sell_px_eff, "executed"


def execute_partitioned(row: pd.Series, pages: pd.DataFrame, balances: dict[str, VenueBalance], cfg: dict, ts_exec_start_ns: int) -> tuple[float, float, float, float, str]:
    parts = int(cfg["strategy"]["partition_count"])
    step_ns = int(float(cfg["strategy"]["partition_step_ms"]) * 1_000_000)
    max_total_qty = float(cfg["strategy"]["max_trade_qty"])
    if parts <= 0:
        return 0.0, 0.0, 0.0, 0.0, "skipped_invalid_partition"

    qty_done = 0.0
    pnl_done = 0.0
    buy_px_last = 0.0
    sell_px_last = 0.0

    for i in range(parts):
        ts_i = ts_exec_start_ns + i * step_ns
        page_i = find_page_at_or_after(ts_i, pages)
        if page_i is None:
            break
        # keep executing only while opportunity is still crossed
        if float(page_i.get("spread", 0.0) or 0.0) <= 0.0:
            break

        rem = max_total_qty - qty_done
        if rem <= 0:
            break
        part_cap = rem / max(parts - i, 1)
        qty_i = feasible_qty(row, balances, cfg, part_cap)
        if qty_i <= 0:
            continue

        buy_px, sell_px, net_i = apply_trade(row, qty_i, balances, cfg)
        buy_px_last, sell_px_last = buy_px, sell_px
        qty_done += qty_i
        pnl_done += net_i

    if qty_done <= 0:
        return 0.0, 0.0, 0.0, 0.0, "skipped_no_capacity"
    if pnl_done < float(cfg["min_net_edge_quote"]):
        return qty_done, buy_px_last, sell_px_last, pnl_done, "skipped_below_threshold"
    return qty_done, buy_px_last, sell_px_last, pnl_done, "executed"


## Path Runner

In [ ]:
def build_initial_balances_for_path(cfg: dict, rng: random.Random) -> dict[str, VenueBalance]:
    mode = str(cfg.get("initial_balance_mode", "fixed"))
    jitter = float(cfg.get("initial_balance_jitter_pct", 0.0))
    if jitter < 0.0:
        raise ValueError("initial_balance_jitter_pct must be >= 0")
    if mode not in {"fixed", "vary_by_path"}:
        raise ValueError(f"Unsupported initial_balance_mode: {mode}")

    balances: dict[str, VenueBalance] = {}
    lo = max(0.0, 1.0 - jitter)
    hi = 1.0 + jitter
    for venue, b in cfg["initial_balances"].items():
        base = float(b["BTC"])
        quote = float(b["USDT"])
        if mode == "vary_by_path" and jitter > 0.0:
            base *= rng.uniform(lo, hi)
            quote *= rng.uniform(lo, hi)
        balances[venue] = VenueBalance(base=base, quote=quote)
    return balances


def run_one_path(path_id: int, cfg: dict, pages: pd.DataFrame, candidates: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = random.Random(seed)
    balances = build_initial_balances_for_path(cfg, rng)
    initial_by_venue = {v: VenueBalance(base=b.base, quote=b.quote) for v, b in balances.items()}
    rows = []
    pnl_cum = 0.0

    for _, cand in candidates.iterrows():
        if cfg["max_candidate_age_ms"] is not None:
            if float(cand.get("buy_age_ms", 0.0)) > float(cfg["max_candidate_age_ms"]):
                rows.append({"path_id": path_id, "ts_book_ns": int(cand["ts_book_ns"]), "status": "skipped_stale", "trade_qty": 0.0, "net_edge_quote": 0.0, "pnl_cum": pnl_cum})
                continue
            if float(cand.get("sell_age_ms", 0.0)) > float(cfg["max_candidate_age_ms"]):
                rows.append({"path_id": path_id, "ts_book_ns": int(cand["ts_book_ns"]), "status": "skipped_stale", "trade_qty": 0.0, "net_edge_quote": 0.0, "pnl_cum": pnl_cum})
                continue

        latency_ms = sample_latency_ms(rng, cfg)
        ts_exec_ns = int(cand["ts_book_ns"] + latency_ms * 1_000_000)
        exec_page = find_page_at_or_after(ts_exec_ns, pages)
        if exec_page is None:
            rows.append({"path_id": path_id, "ts_book_ns": int(cand["ts_book_ns"]), "ts_exec_ns": ts_exec_ns, "status": "skipped_no_exec_page", "trade_qty": 0.0, "net_edge_quote": 0.0, "pnl_cum": pnl_cum})
            continue

        if str(cfg["strategy"]["name"]) == "all_at_once":
            qty, buy_px_eff, sell_px_eff, status = execute_all_at_once(cand, exec_page, balances, cfg)
            net = 0.0
            if status == "executed":
                # Reconstruct net from balance delta is overkill; compute from effective px.
                mid = (float(exec_page["best_ask"]) + float(exec_page["best_bid"])) / 2.0 if exec_page.get("best_ask") is not None and exec_page.get("best_bid") is not None else (float(cand["buy_price"]) + float(cand["sell_price"])) / 2.0
                net = (sell_px_eff - buy_px_eff) * qty - (mid * qty * (float(cfg["latency_haircut_bps"]) / 10000.0)) - float(cfg["fixed_cost_quote"])
        else:
            qty, buy_px_eff, sell_px_eff, net, status = execute_partitioned(cand, pages, balances, cfg, ts_exec_ns)

        if status == "executed":
            pnl_cum += float(net)

        rows.append({
            "path_id": path_id,
            "ts_book_ns": int(cand["ts_book_ns"]),
            "ts_exec_ns": ts_exec_ns,
            "latency_ms": latency_ms,
            "buy_venue": str(cand["buy_venue"]),
            "sell_venue": str(cand["sell_venue"]),
            "status": status,
            "trade_qty": float(qty),
            "buy_px_eff": float(buy_px_eff),
            "sell_px_eff": float(sell_px_eff),
            "net_edge_quote": float(net),
            "pnl_cum": float(pnl_cum),
        })

    path_df = pd.DataFrame(rows)
    bal_df = pd.DataFrame([
        {
            "path_id": path_id,
            "venue": v,
            "initial_base": initial_by_venue[v].base,
            "initial_quote": initial_by_venue[v].quote,
            "base": b.base,
            "quote": b.quote,
        }
        for v, b in sorted(balances.items())
    ])
    return path_df, bal_df


candidates_df = normalize_candidate_schema(candidates_df)
required_sim_cols = ["ts_book_ns", "buy_venue", "sell_venue", "buy_price", "sell_price", "buy_qty", "sell_qty"]
missing_sim_cols = [c for c in required_sim_cols if c not in candidates_df.columns]
if missing_sim_cols:
    raise KeyError(
        f"Simulation requires columns {required_sim_cols}, missing={missing_sim_cols}. "
        f"Available={list(candidates_df.columns)}. Re-run the data loading cell."
    )

all_path_rows = []
all_balance_rows = []
for path_id in range(int(SIM_CONFIG["n_paths"])):
    seed = int(SIM_CONFIG["base_seed"]) + path_id
    p_df, b_df = run_one_path(path_id, SIM_CONFIG, pages_df, candidates_df, seed)
    all_path_rows.append(p_df)
    all_balance_rows.append(b_df)

sim_paths_df = pd.concat(all_path_rows, ignore_index=True) if all_path_rows else pd.DataFrame()
sim_balances_df = pd.concat(all_balance_rows, ignore_index=True) if all_balance_rows else pd.DataFrame()

if not sim_paths_df.empty:
    sim_paths_df["ts_exec_dt"] = pd.to_datetime(sim_paths_df["ts_exec_ns"], unit="ns", errors="coerce")

{
    "paths": int(SIM_CONFIG["n_paths"]),
    "rows": int(len(sim_paths_df)),
    "executed": int((sim_paths_df["status"] == "executed").sum()) if not sim_paths_df.empty else 0,
    "skipped": int((sim_paths_df["status"] != "executed").sum()) if not sim_paths_df.empty else 0,
}


## Results Preview

In [ ]:
sim_paths_df.head(20) if not sim_paths_df.empty else "No simulation rows."

In [ ]:
sim_balances_df.head(20) if not sim_balances_df.empty else "No balance rows."

## Path Summary

In [ ]:
if sim_paths_df.empty:
    "No simulation rows."
else:
    sim_paths_df.groupby("path_id").agg(
        executed=("status", lambda s: int((s == "executed").sum())),
        skipped=("status", lambda s: int((s != "executed").sum())),
        final_pnl_quote=("pnl_cum", "last"),
        avg_latency_ms=("latency_ms", "mean"),
    ).sort_values("final_pnl_quote", ascending=False)


## Yield Curve (Per Path)

Plot cumulative quote PnL over execution time for all paths.

In [ ]:
if sim_paths_df.empty:
    "No simulation rows."
else:
    plt.figure(figsize=(12, 5))
    for path_id, g in sim_paths_df.sort_values("ts_exec_ns").groupby("path_id"):
        plt.plot(g["ts_exec_dt"], g["pnl_cum"], alpha=0.35)
    plt.title("Arbitrage Run Yield Curves (Quote PnL)")
    plt.xlabel("Execution time")
    plt.ylabel("Cumulative PnL (quote)")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()